# A2: CLIP no ADS-16 (ranking de objetos e busca por texto)

**Requisitos (PD: memória e tempo).** Memória: CLIP ViT-B/32 ocupa cerca de 0,6 GB de pesos; cabe numa T4 (16 GB) e no CPU. Tempo estimado: cerca de 1 min com os embeddings já no DuckDB; na primeira vez, mais cerca de 15 s para gerar os 600 embeddings no Apple M5 (MPS) e alguns minutos de download dos modelos (CLIP e `bert-base-cased`).

**O que este notebook faz.** Ele é uma vitrine: importa o pacote `visaocomp` e chama as mesmas classes de `scripts/run_a2.py` (`Ads16Sampler`, `ClipEmbedder`, `EmbeddingStore`, `ObjectRanker`, `TextSearch`). Nenhum modelo é treinado. Todo número vem de `outputs/<run_id>/a2/`.

**Na Colab.** Copie a pasta `entrega/` para o Drive, monte o Drive, rode `pip install -e .` nela e `make data` (baixa o ADS-16), e depois execute as células abaixo.

**Licença do ADS-16.** Só uso científico; as imagens não podem ser publicadas. As figuras ficam em `outputs/`, fora do git, e o notebook salvo no repositório não guarda as imagens nas saídas (rode as células para vê-las).

**Licença do ADS-16:** o dataset é só para uso científico e as imagens não podem ser publicadas. Por isso, nesta cópia pública, as saídas com imagens dos anúncios foram retiradas; as tabelas e os textos ficam. Rodando o notebook com o dataset baixado (`make data`), as imagens aparecem de novo.

In [1]:
import logging
import warnings

warnings.filterwarnings("ignore")
import pandas as pd
from IPython.display import Image as Show, Markdown, display

from visaocomp.a2.config import load_a2_config
from visaocomp.a2.pipeline import build_pipeline
from visaocomp.config import default_config_path
from visaocomp.log import configure_logging
from visaocomp.paths import data_dir, repo_root
from visaocomp.run import latest_manifest

configure_logging()
for noisy in ("httpx", "httpx2", "httpcore", "huggingface_hub", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
cfg = load_a2_config()
root = repo_root()
print("dados em", data_dir())

dados em ./data.nosync


## 1. Executar a etapa (mesmas classes do `make run-a2`)

O que já está no DuckDB não é recalculado: veja a linha `embeddings: N novos, M reaproveitados`.

In [2]:
pipe = build_pipeline(cfg, data_dir(), root / "outputs", root / cfg.threshold_check_file,
                      data_cfg_path=default_config_path())
run = pipe.run()
out = run.path / "a2"
print(run.run_id)

23:58:18 INFO visaocomp.a2.pipeline: A2: início, run 20261013-025818-a2


23:58:18 INFO visaocomp.a2.sampler: ADS-16: 301 anúncios em 20 pastas, 2396 fotos pessoais; pastas fora de 15: {'1': 16}


23:58:18 INFO visaocomp.a2.sampler: amostra: 600 imagens (301 anúncios, 299 fotos pessoais), seed 42


23:58:18 INFO visaocomp.a2.pipeline: embeddings: 0 novos, 600 reaproveitados, 0 ilegíveis


23:58:20 INFO visaocomp.a2.embedder: carregando openai/clip-vit-base-patch32 em mps


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10354.38it/s]


[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


23:58:25 INFO visaocomp.a2.pipeline: A2: fim em 7.0s; ./outputs/20261013-025818-a2


20261013-025818-a2


## 2. Dados: mapa pasta para categoria e amostra

In [3]:
import json
cat = json.loads((out / "category_map.json").read_text())
display(pd.DataFrame({"categoria": cat["folder_to_category"], "anúncios": cat["ads_per_folder"]}))
sample = pd.read_csv(out / "sample.csv")
print(len(sample), "imagens;", sample.source.value_counts().to_dict())

,categoria,anúncios
1,Clothing & Shoes,16
2,Automotive,15
3,Baby Products,15
4,Health & Beauty,15
5,Media (BMVD),15
6,Consumer Electronics,15
7,Console & Video Games,15
8,DIY & Tools,15
9,Garden & Outdoor living,15
10,Grocery,15


600 imagens; {'ad': 301, 'pessoal': 299}


## 3. R18: limiar, ranking e os 5 mais frequentes

In [4]:
thr = json.loads((out / "threshold.json").read_text())
print(f"limiar = {thr['threshold']:.3f}  (acerta {thr['accuracy']:.0%} dos {thr['n_labeled']} pares conferidos a olho)")
ranking = pd.read_csv(out / "ranking.csv")
display(ranking.round(3))

limiar = 0.250  (acerta 76% dos 50 pares conferidos a olho)


,rank,description,n_present,frequency,mean_score,mean_score_present,n_top1,frequency_ad,frequency_pessoal
0,1,a website screenshot,138,0.230,0.231,0.271,114,0.435,0.023
1,2,text and logo,63,0.105,0.219,0.261,38,0.176,0.033
2,3,food product,55,0.092,0.214,0.264,54,0.090,0.094
3,4,a video game,49,0.082,0.219,0.264,32,0.126,0.037
4,5,a pet animal,32,0.053,0.205,0.263,37,0.023,0.084
5,6,a toy,31,0.052,0.214,0.260,26,0.076,0.027
6,7,a person,28,0.047,0.214,0.259,35,0.023,0.070
7,8,a baby,24,0.040,0.206,0.274,23,0.050,0.030
8,9,clothing or shoes,23,0.038,0.203,0.266,21,0.063,0.013
9,10,"a book, DVD or CD",22,0.037,0.209,0.263,27,0.056,0.017


In [5]:
display(Show(filename=str(out / "top5.png")))

## 4. R19: busca por texto (top-5 por consulta)

In [6]:
queries = json.loads((out / "queries.json").read_text())
for q in queries:
    print(q["id"], q["text"], "|", q["nivel"])
    display(Show(filename=str(out / "queries" / f"{q['id']}.png")))

q01 food | genérico, concreto


q02 a bowl of fresh fruit and vegetables | específico, concreto


q03 a red sports car on a road | específico, concreto


q04 a cute puppy | específico, concreto


q05 a person using a smartphone | específico, relação entre objetos


q06 a big discount sale offer | específico, texto na imagem


q07 a romantic couple | intermediário, emoção


q08 a cold drink on a hot summer day | específico, cena e sensação


q09 happiness | abstrato


q10 luxury | abstrato


Uma consulta nova, ao vivo (cosseno dentro do DuckDB):

In [7]:
from visaocomp.a2.search import TextSearch
live = TextSearch(pipe.embedder, pipe.store, list(sample.image_path), cfg.top_k).query("a dog playing outside")
display(pd.DataFrame(live))

,rank,image_path,score,source,category
0,1,Corpus/U0079/U0079-IM-NEG/3.png,0.276775,pessoal,
1,2,Corpus/U0012/U0012-IM-NEG/3_th_1445459433.png,0.262062,pessoal,
2,3,Corpus/U0118/U0118-IM-NEG/2_th_1459151531.png,0.248365,pessoal,
3,4,Corpus/U0086/U0086-IM-NEG/3.png,0.236486,pessoal,
4,5,Corpus/U0071/U0071-IM-POS/4.png,0.235568,pessoal,


## 5. R17: o alinhamento medido

In [8]:
ali = json.loads((out / "alignment.json").read_text())
display(pd.Series({k: v for k, v in ali.items() if k != "per_category_top1"}, name="valor").to_frame())
display(pd.Series(ali["per_category_top1"], name="top-1 por categoria").sort_values(ascending=False).to_frame())

,valor
n_ads,301.000000
n_categories,20.000000
zero_shot_top1,0.548173
zero_shot_top5,0.797342
chance_top1,0.050000
chance_top5,0.250000
cos_image_vs_true_category_text,0.250468
cos_image_vs_other_category_text,0.209135
cos_image_image,0.419417
cos_text_text,0.846650


,top-1 por categoria
Console & Video Games,1.000000
Baby Products,0.866667
Consumer Electronics,0.866667
Dating Sites,0.800000
Betting,0.800000
Jewellery & Watches,0.800000
Pet Supplies,0.733333
DIY & Tools,0.733333
Grocery,0.733333
Office Products,0.600000


## 6. R20: os dois tokenizadores e a máscara

In [9]:
tok = json.loads((out / "tokenizers.json").read_text())
for c, b in zip(tok["clip"]["tokens"], tok["bert"]["tokens"]):
    print("CLIP:", c["tokens"], "| reais", c["n_real"], "padding", c["n_padding"])
    print("BERT:", b["tokens"], "| reais", b["n_real"], "padding", b["n_padding"])
    print()
print("CLIP:", tok["clip"]["mask_experiment"])
print("BERT:", tok["bert"]["mask_experiment"])

CLIP: ['<|startoftext|>', 'a</w>', 'car</w>', '<|endoftext|>'] | reais 4 padding 73
BERT: ['[CLS]', 'a', 'car', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'] | reais 4 padding 12

CLIP: ['<|startoftext|>', 'a</w>', 'red</w>', 'sports</w>', 'car</w>', 'on</w>', 'a</w>', 'road</w>', '<|endoftext|>'] | reais 9 padding 68
BERT: ['[CLS]', 'a', 'red', 'sports', 'car', 'on', 'a', 'road', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'] | reais 9 padding 7

CLIP: ['<|startoftext|>', 'a</w>', 'person</w>', 'using</w>', 'a</w>', 'smartphone</w>', 'at</w>', 'the</w>', 'beach</w>', 'while</w>', 'drinking</w>', 'a</w>', 'cold</w>', 'drink</w>', '<|endoftext|>'] | reais 15 padding 62
BERT: ['[CLS]', 'a', 'person', 'using', 'a', 'smart', '##phone', 'at', 'the', 'beach', 'while', 'drinking', 'a', 'cold', 'drink', '[SEP]'] | reais 16 padding 0

CLIP: {'sentence': 'a car', 'cos_alone_vs_padded_with_mask': 1.0, 

O texto completo da análise está em `reports/sections/a2.md` (gerado por `python scripts/run_a2.py --report-only`).